# ActiveMQ Artemis: 3-node Horizontal Scaling Cluster with Docker Compose

This notebook turns the previous answer into a copy/paste-friendly guide.

## Goal
- Run **three ActiveMQ Artemis brokers** in a **cluster** (horizontal scaling)
- Use **static connectors** (reliable in Docker)
- Expose each broker on a different host port
- Provide failover URLs for clients

## What this is (and isn't)
- ✅ Great for scaling **consumer throughput** (competing consumers across nodes)
- ✅ Clients can connect to **any node**
- ❌ Not **HA active/passive** (if one node dies, it’s gone unless you add HA)
- ❌ Not "replicate every queue everywhere" (each queue/address has a home; clustering forwards as needed)


## 1) Project layout

Create this folder structure:

```
artemis-3node-cluster/
  docker-compose.yml
  conf/
    broker1/broker.xml
    broker2/broker.xml
    broker3/broker.xml
```


## 2) `docker-compose.yml`

Save as `artemis-3node-cluster/docker-compose.yml`:


In [ ]:
docker_compose_yml = r'''version: "3.8"

networks:
  artemis-net:

volumes:
  broker1-data:
  broker2-data:
  broker3-data:

services:
  broker1:
    image: apache/activemq-artemis:latest
    container_name: broker1
    hostname: broker1
    environment:
      ARTEMIS_USER: admin
      ARTEMIS_PASSWORD: admin
      ANONYMOUS_LOGIN: "false"
    volumes:
      - broker1-data:/var/lib/artemis-instance
      - ./conf/broker1/broker.xml:/var/lib/artemis-instance/etc/broker.xml:ro
    ports:
      - "61616:61616" # CORE
      - "8161:8161"   # console
    networks: [artemis-net]

  broker2:
    image: apache/activemq-artemis:latest
    container_name: broker2
    hostname: broker2
    environment:
      ARTEMIS_USER: admin
      ARTEMIS_PASSWORD: admin
      ANONYMOUS_LOGIN: "false"
    volumes:
      - broker2-data:/var/lib/artemis-instance
      - ./conf/broker2/broker.xml:/var/lib/artemis-instance/etc/broker.xml:ro
    ports:
      - "61617:61616"
      - "8162:8161"
    networks: [artemis-net]

  broker3:
    image: apache/activemq-artemis:latest
    container_name: broker3
    hostname: broker3
    environment:
      ARTEMIS_USER: admin
      ARTEMIS_PASSWORD: admin
      ANONYMOUS_LOGIN: "false"
    volumes:
      - broker3-data:/var/lib/artemis-instance
      - ./conf/broker3/broker.xml:/var/lib/artemis-instance/etc/broker.xml:ro
    ports:
      - "61618:61616"
      - "8163:8161"
    networks: [artemis-net]
'''
print(docker_compose_yml)


## 3) `broker.xml` configs

Each broker gets a `broker.xml` with:
- a unique `<name>`
- an acceptor for `CORE,AMQP` (optional but convenient)
- connectors pointing at **all nodes**
- a `cluster-connection` with `static-connectors`

> These files are **minimal** and focus on clustering blocks. In production, start from the default `broker.xml` and merge these sections.


### `conf/broker1/broker.xml`


In [ ]:
broker1_xml = r'''<?xml version="1.0"?>
<configuration xmlns="urn:activemq"
               xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
               xsi:schemaLocation="urn:activemq /schema/artemis-configuration.xsd">

  <core xmlns="urn:activemq:core">
    <name>broker1</name>

    <acceptors>
      <acceptor name="core">tcp://0.0.0.0:61616?protocols=CORE,AMQP</acceptor>
    </acceptors>

    <connectors>
      <connector name="c1">tcp://broker1:61616</connector>
      <connector name="c2">tcp://broker2:61616</connector>
      <connector name="c3">tcp://broker3:61616</connector>
    </connectors>

    <cluster-connections>
      <cluster-connection name="cluster">
        <connector-ref>c1</connector-ref>
        <retry-interval>1000</retry-interval>
        <use-duplicate-detection>true</use-duplicate-detection>
        <message-load-balancing>ON_DEMAND</message-load-balancing>

        <static-connectors>
          <connector-ref>c2</connector-ref>
          <connector-ref>c3</connector-ref>
        </static-connectors>
      </cluster-connection>
    </cluster-connections>
  </core>
</configuration>
'''
print(broker1_xml)


### `conf/broker2/broker.xml`


In [ ]:
broker2_xml = r'''<?xml version="1.0"?>
<configuration xmlns="urn:activemq"
               xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
               xsi:schemaLocation="urn:activemq /schema/artemis-configuration.xsd">

  <core xmlns="urn:activemq:core">
    <name>broker2</name>

    <acceptors>
      <acceptor name="core">tcp://0.0.0.0:61616?protocols=CORE,AMQP</acceptor>
    </acceptors>

    <connectors>
      <connector name="c1">tcp://broker1:61616</connector>
      <connector name="c2">tcp://broker2:61616</connector>
      <connector name="c3">tcp://broker3:61616</connector>
    </connectors>

    <cluster-connections>
      <cluster-connection name="cluster">
        <connector-ref>c2</connector-ref>
        <retry-interval>1000</retry-interval>
        <use-duplicate-detection>true</use-duplicate-detection>
        <message-load-balancing>ON_DEMAND</message-load-balancing>

        <static-connectors>
          <connector-ref>c1</connector-ref>
          <connector-ref>c3</connector-ref>
        </static-connectors>
      </cluster-connection>
    </cluster-connections>
  </core>
</configuration>
'''
print(broker2_xml)


### `conf/broker3/broker.xml`


In [ ]:
broker3_xml = r'''<?xml version="1.0"?>
<configuration xmlns="urn:activemq"
               xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
               xsi:schemaLocation="urn:activemq /schema/artemis-configuration.xsd">

  <core xmlns="urn:activemq:core">
    <name>broker3</name>

    <acceptors>
      <acceptor name="core">tcp://0.0.0.0:61616?protocols=CORE,AMQP</acceptor>
    </acceptors>

    <connectors>
      <connector name="c1">tcp://broker1:61616</connector>
      <connector name="c2">tcp://broker2:61616</connector>
      <connector name="c3">tcp://broker3:61616</connector>
    </connectors>

    <cluster-connections>
      <cluster-connection name="cluster">
        <connector-ref>c3</connector-ref>
        <retry-interval>1000</retry-interval>
        <use-duplicate-detection>true</use-duplicate-detection>
        <message-load-balancing>ON_DEMAND</message-load-balancing>

        <static-connectors>
          <connector-ref>c1</connector-ref>
          <connector-ref>c2</connector-ref>
        </static-connectors>
      </cluster-connection>
    </cluster-connections>
  </core>
</configuration>
'''
print(broker3_xml)


## 4) Start the cluster

From inside `artemis-3node-cluster/`:

```bash
docker compose up -d
docker logs -f broker1
docker logs -f broker2
docker logs -f broker3
```


## 5) Client connection URLs

### From the host machine
Ports are mapped as:
- broker1: `localhost:61616`
- broker2: `localhost:61617`
- broker3: `localhost:61618`

**CORE/JMS failover URL:**

```
failover:(tcp://localhost:61616,tcp://localhost:61617,tcp://localhost:61618)
```

### From another container on the same Compose network

```
failover:(tcp://broker1:61616,tcp://broker2:61616,tcp://broker3:61616)
```


## 6) Notes on horizontal scaling expectations

- This setup helps you scale **consumer throughput** by distributing consumers across nodes.
- A single hot queue can still become a bottleneck. For true parallelism, consider **partitioning**:
  - `orders.0 ... orders.15`
  - route by a key (hash of customerId, tenantId, etc.)
- For **high availability**, pair each node with a backup (Replication HA) or use another HA strategy.


## 7) Optional: generate the files from this notebook

Run the next cell to create the folder structure and write all files to disk.


In [ ]:
from pathlib import Path

base = Path('artemis-3node-cluster')
(base / 'conf' / 'broker1').mkdir(parents=True, exist_ok=True)
(base / 'conf' / 'broker2').mkdir(parents=True, exist_ok=True)
(base / 'conf' / 'broker3').mkdir(parents=True, exist_ok=True)

(base / 'docker-compose.yml').write_text(docker_compose_yml, encoding='utf-8')
(base / 'conf' / 'broker1' / 'broker.xml').write_text(broker1_xml, encoding='utf-8')
(base / 'conf' / 'broker2' / 'broker.xml').write_text(broker2_xml, encoding='utf-8')
(base / 'conf' / 'broker3' / 'broker.xml').write_text(broker3_xml, encoding='utf-8')

print(f"Wrote files under: {base.resolve()}")
